# 11 — Improved Tuning: Temporal Decay Weights + More Trials

**Changes vs notebook 07:**
- Dataset: `updated_model_dataset.csv` (all available data)
- Temporal decay weights per CV fold:
  - Matches in the WC year (before WC, already excluded from val): **×1.8**
  - Matches in year before WC: **×1.4**
  - All other matches: **×1.0**
  - Multiplied on top of competition weights, then re-normalized
- Optuna trials: **300** per model (up from 150)
- Same CV strategy: 3 folds (2014, 2018, 2022 WC)
- Same objective: maximize avg exact score accuracy across all 3 folds

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from src.models.base import load_model_dataset
from src.features.feature_columns import FEATURE_COLS
from src.models.world_cup_utils import create_wc_cv_splits
from src.models.optuna_tuning import (
    LGBMTuner, XGBTuner, WCCVTuner, exact_score_accuracy, result_accuracy
)
from src.models.lgbm_model import LGBMGoalModel
from src.models.xgb_model import XGBGoalModel
from src.models.poisson_model import PoissonGoalModel
from src.models.ensemble import EnsembleGoalModel
from src.models.score_conversion import win_draw_loss_probs
from src.evaluation.metrics import rps_batch
from src.models.weighting import apply_competition_weights

TARGET_COLS = ['goals_A', 'goals_B']
CV_YEARS = [2014, 2018, 2022]

print('Imports OK')

Imports OK


## 1. Load Updated Dataset

In [2]:
df = load_model_dataset('../data/processed/updated_model_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Dataset: {len(df)} matches, {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Features: {len(FEATURE_COLS)}')

missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'WARNING: Missing features: {missing}')
else:
    print('All production features present ✓')

# Check WC years available
wc_df = df[df['competition'].str.strip().str.lower() == 'world cup']
print(f'WC matches by year: {wc_df["tournament_year"].value_counts().sort_index().to_dict()}')

Dataset: 21655 matches, 2004-01-01 to 2026-06-08
Features: 20
All production features present ✓
WC matches by year: {2006: 64, 2010: 64, 2014: 64, 2018: 64, 2022: 64}


## 2. Per-Fold Temporal × Competition Weights

For each CV fold (WC year Y), we weight training matches as:
- Year == Y (same year as WC, non-WC matches): ×1.8
- Year == Y-1 (year before WC): ×1.4  
- All others: ×1.0

This is multiplied on top of competition weights, then normalized to mean=1.

In [3]:
def temporal_multiplier(train_df: pd.DataFrame, wc_year: int) -> np.ndarray:
    """Return per-row temporal multiplier relative to a WC year."""
    years = train_df['date'].dt.year.values
    mult = np.ones(len(train_df))
    mult[years == wc_year] = 1.8      # WC year (already before WC since WC matches are in val)
    mult[years == wc_year - 1] = 1.4  # Year before WC
    return mult


def build_fold_weights(df: pd.DataFrame, wc_year: int) -> np.ndarray:
    """Combine competition weights × temporal decay for a given WC fold."""
    val_mask = (
        (df['tournament_year'] == wc_year) &
        (df['competition'].str.strip().str.lower() == 'world cup')
    )
    train_df = df[~val_mask].reset_index(drop=True)

    comp_w = apply_competition_weights(train_df)          # normalized to mean=1
    temp_m = temporal_multiplier(train_df, wc_year)       # fold-specific multiplier
    combined = comp_w * temp_m
    combined = combined / combined.mean()                  # renormalize
    return combined


# Create CV splits (features + targets)
cv_splits = create_wc_cv_splits(df, CV_YEARS, FEATURE_COLS, TARGET_COLS)

# Build per-fold weights
weights_per_fold = {}
for year in CV_YEARS:
    w = build_fold_weights(df, year)
    weights_per_fold[year] = w
    X_tr, _, _, _ = cv_splits[year]
    print(f'WC {year}: weight shape={w.shape}, mean={w.mean():.3f}, '
          f'WC-year train rows={int((df[~((df["tournament_year"]==year) & (df["competition"].str.strip().str.lower()=="world cup"))]["date"].dt.year == year).sum())} '
          f'(×1.8), prev-year rows={(df[~((df["tournament_year"]==year) & (df["competition"].str.strip().str.lower()=="world cup"))]["date"].dt.year == year - 1).sum()} (×1.4)')

WC 2014 fold: train=21591, val=64 (WC only)
WC 2018 fold: train=21591, val=64 (WC only)
WC 2022 fold: train=21591, val=64 (WC only)
WC 2014: weight shape=(21591,), mean=1.000, WC-year train rows=817 (×1.8), prev-year rows=954 (×1.4)
WC 2018: weight shape=(21591,), mean=1.000, WC-year train rows=834 (×1.8), prev-year rows=886 (×1.4)
WC 2022: weight shape=(21591,), mean=1.000, WC-year train rows=923 (×1.8), prev-year rows=1133 (×1.4)


## 3. Baseline — Untuned Models

In [4]:
def eval_model_cv(model_class, cv_splits, weights_per_fold=None, **kwargs):
    """Evaluate a model on all CV folds and return per-fold + avg metrics."""
    rows = []
    for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
        w = (weights_per_fold or {}).get(year)
        model = model_class(**kwargs)
        model.fit(X_tr, y_tr, sample_weight=w)
        y_pred = np.clip(model.predict(X_val), 0, None)
        exact = exact_score_accuracy(y_val, y_pred)
        result = result_accuracy(y_val, y_pred)
        probs = np.array([win_draw_loss_probs(a, b) for a, b in y_pred])
        rps = rps_batch(y_val, probs)
        rows.append({'year': year, 'exact': exact, 'result': result, 'rps': rps, 'n': len(y_val)})

    result_df = pd.DataFrame(rows)
    avg = result_df[['exact', 'result', 'rps']].mean()
    avg_row = pd.DataFrame([{'year': 'avg', 'exact': avg['exact'], 'result': avg['result'], 'rps': avg['rps'], 'n': ''}])
    return pd.concat([result_df, avg_row], ignore_index=True)


print('=== Poisson (default) ===')
poisson_baseline = eval_model_cv(PoissonGoalModel, cv_splits, weights_per_fold)
print(poisson_baseline[['year', 'exact', 'result', 'rps']].to_string(index=False))

print('\n=== LightGBM (default params) ===')
lgbm_baseline = eval_model_cv(LGBMGoalModel, cv_splits, weights_per_fold)
print(lgbm_baseline[['year', 'exact', 'result', 'rps']].to_string(index=False))

print('\n=== XGBoost (default params) ===')
xgb_baseline = eval_model_cv(XGBGoalModel, cv_splits, weights_per_fold)
print(xgb_baseline[['year', 'exact', 'result', 'rps']].to_string(index=False))

=== Poisson (default) ===
year    exact   result      rps
2014 0.234375 0.765625 0.133453
2018 0.218750 0.781250 0.127327
2022 0.171875 0.718750 0.141870
 avg 0.208333 0.755208 0.134216

=== LightGBM (default params) ===
year    exact   result      rps
2014 0.218750 0.937500 0.070768
2018 0.312500 0.890625 0.064242
2022 0.265625 0.937500 0.070288
 avg 0.265625 0.921875 0.068432

=== XGBoost (default params) ===
year    exact   result      rps
2014 0.218750 0.953125 0.070950
2018 0.281250 0.890625 0.059570
2022 0.234375 0.937500 0.069574
 avg 0.244792 0.927083 0.066698


## 4. Tune LightGBM — 300 Trials

In [5]:
N_TRIALS = 300
TIMEOUT  = 3900  # 1 hour max per model

print(f'Tuning LightGBM over {N_TRIALS} trials (CV across {CV_YEARS})...')
lgbm_cv_tuner = WCCVTuner(
    base_tuner_class=LGBMTuner,
    cv_splits=cv_splits,
    weights_per_fold=weights_per_fold,
)
lgbm_result = lgbm_cv_tuner.optimize(n_trials=N_TRIALS, timeout=TIMEOUT, verbose=True)

Tuning LightGBM over 300 trials (CV across [2014, 2018, 2022])...


  0%|          | 0/300 [00:00<?, ?it/s]


Best avg exact score accuracy across folds: 0.2917
Best parameters: {'n_estimators': 461, 'max_depth': 12, 'learning_rate': 0.025037651451202188, 'num_leaves': 200, 'min_child_samples': 44, 'subsample': 0.8033348792290621, 'colsample_bytree': 0.6203538805551276, 'reg_alpha': 7.910467580693931e-06, 'reg_lambda': 0.03337081102245976}


In [6]:
print('Best LightGBM hyperparameters:')
for k, v in lgbm_result['best_params'].items():
    print(f'  {k}: {v}')
print(f'\nBest avg exact score accuracy: {lgbm_result["best_avg_acc"]*100:.2f}%')

print('\nPer-fold evaluation (tuned LightGBM):')
for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
    w = weights_per_fold.get(year)
    m = LGBMGoalModel(**lgbm_result['best_params'])
    m.fit(X_tr, y_tr, sample_weight=w)
    y_pred = np.clip(m.predict(X_val), 0, None)
    exact = exact_score_accuracy(y_val, y_pred)
    result = result_accuracy(y_val, y_pred)
    probs = np.array([win_draw_loss_probs(a, b) for a, b in y_pred])
    rps = rps_batch(y_val, probs)
    print(f'  WC {year}: exact={exact*100:.1f}%  result={result*100:.1f}%  RPS={rps:.4f}')

Best LightGBM hyperparameters:
  n_estimators: 461
  max_depth: 12
  learning_rate: 0.025037651451202188
  num_leaves: 200
  min_child_samples: 44
  subsample: 0.8033348792290621
  colsample_bytree: 0.6203538805551276
  reg_alpha: 7.910467580693931e-06
  reg_lambda: 0.03337081102245976

Best avg exact score accuracy: 29.17%

Per-fold evaluation (tuned LightGBM):
  WC 2014: exact=25.0%  result=90.6%  RPS=0.0765
  WC 2018: exact=32.8%  result=90.6%  RPS=0.0645
  WC 2022: exact=29.7%  result=93.8%  RPS=0.0750


## 5. Tune XGBoost — 300 Trials

In [7]:
print(f'Tuning XGBoost over {N_TRIALS} trials (CV across {CV_YEARS})...')
xgb_cv_tuner = WCCVTuner(
    base_tuner_class=XGBTuner,
    cv_splits=cv_splits,
    weights_per_fold=weights_per_fold,
)
xgb_result = xgb_cv_tuner.optimize(n_trials=N_TRIALS, timeout=TIMEOUT, verbose=True)

Tuning XGBoost over 300 trials (CV across [2014, 2018, 2022])...


  0%|          | 0/300 [00:00<?, ?it/s]


Best avg exact score accuracy across folds: 0.2812
Best parameters: {'n_estimators': 480, 'max_depth': 12, 'learning_rate': 0.015763210575425127, 'subsample': 0.8995970132570389, 'colsample_bytree': 0.6946673112765889, 'gamma': 4.126453395664713, 'min_child_weight': 19, 'reg_alpha': 0.012107938073992418, 'reg_lambda': 4.164462463380124e-06}


In [8]:
print('Best XGBoost hyperparameters:')
for k, v in xgb_result['best_params'].items():
    print(f'  {k}: {v}')
print(f'\nBest avg exact score accuracy: {xgb_result["best_avg_acc"]*100:.2f}%')

print('\nPer-fold evaluation (tuned XGBoost):')
for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
    w = weights_per_fold.get(year)
    m = XGBGoalModel(**xgb_result['best_params'])
    m.fit(X_tr, y_tr, sample_weight=w)
    y_pred = np.clip(m.predict(X_val), 0, None)
    exact = exact_score_accuracy(y_val, y_pred)
    result = result_accuracy(y_val, y_pred)
    probs = np.array([win_draw_loss_probs(a, b) for a, b in y_pred])
    rps = rps_batch(y_val, probs)
    print(f'  WC {year}: exact={exact*100:.1f}%  result={result*100:.1f}%  RPS={rps:.4f}')

Best XGBoost hyperparameters:
  n_estimators: 480
  max_depth: 12
  learning_rate: 0.015763210575425127
  subsample: 0.8995970132570389
  colsample_bytree: 0.6946673112765889
  gamma: 4.126453395664713
  min_child_weight: 19
  reg_alpha: 0.012107938073992418
  reg_lambda: 4.164462463380124e-06

Best avg exact score accuracy: 28.12%

Per-fold evaluation (tuned XGBoost):
  WC 2014: exact=26.6%  result=92.2%  RPS=0.0825
  WC 2018: exact=34.4%  result=93.8%  RPS=0.0694
  WC 2022: exact=23.4%  result=92.2%  RPS=0.0846


## 6. Ensemble Weight Search

In [9]:
weight_grid = []
for w_lgb in np.arange(0.0, 1.05, 0.1):
    for w_xgb in np.arange(0.0, 1.05 - w_lgb, 0.1):
        w_pois = round(1.0 - w_lgb - w_xgb, 2)
        if w_pois < -0.01:
            continue
        weight_grid.append((round(w_lgb, 2), round(w_xgb, 2), max(w_pois, 0.0)))

print(f'Testing {len(weight_grid)} weight combinations...')

best_ensemble = {'avg_exact': 0.0, 'w_lgb': 0, 'w_xgb': 0, 'w_pois': 0}
ensemble_results = []

for w_lgb, w_xgb, w_pois in weight_grid:
    if w_lgb + w_xgb + w_pois < 0.99:
        continue
    fold_exacts = []
    for year, (X_tr, y_tr, X_val, y_val) in cv_splits.items():
        wt = weights_per_fold.get(year)
        models, wts = [], []
        if w_lgb > 0:
            m = LGBMGoalModel(**lgbm_result['best_params'])
            m.fit(X_tr, y_tr, sample_weight=wt)
            models.append(m); wts.append(w_lgb)
        if w_xgb > 0:
            m = XGBGoalModel(**xgb_result['best_params'])
            m.fit(X_tr, y_tr, sample_weight=wt)
            models.append(m); wts.append(w_xgb)
        if w_pois > 0:
            m = PoissonGoalModel()
            m.fit(X_tr, y_tr, sample_weight=wt)
            models.append(m); wts.append(w_pois)
        ensemble = EnsembleGoalModel(models, weights=wts)
        y_pred = np.clip(ensemble.predict(X_val), 0, None)
        fold_exacts.append(exact_score_accuracy(y_val, y_pred))
    avg = np.mean(fold_exacts)
    ensemble_results.append({'w_lgb': w_lgb, 'w_xgb': w_xgb, 'w_pois': w_pois, 'avg_exact': avg})
    if avg > best_ensemble['avg_exact']:
        best_ensemble = {'avg_exact': avg, 'w_lgb': w_lgb, 'w_xgb': w_xgb, 'w_pois': w_pois}

ens_df = pd.DataFrame(ensemble_results).sort_values('avg_exact', ascending=False)
print('Top 10 ensemble configurations:')
print(ens_df.head(10).to_string(index=False))
print(f'\nChampion: LGB={best_ensemble["w_lgb"]} XGB={best_ensemble["w_xgb"]} Pois={best_ensemble["w_pois"]}')
print(f'Avg exact score: {best_ensemble["avg_exact"]*100:.2f}%')

Testing 66 weight combinations...
Top 10 ensemble configurations:
 w_lgb  w_xgb  w_pois  avg_exact
   1.0    0.0     0.0   0.291667
   0.0    1.0     0.0   0.281250
   0.1    0.9     0.0   0.276042
   0.6    0.3     0.1   0.270833
   0.2    0.8     0.0   0.270833
   0.3    0.6     0.1   0.270833
   0.1    0.5     0.4   0.270833
   0.2    0.4     0.4   0.270833
   0.5    0.4     0.1   0.270833
   0.0    0.6     0.4   0.270833

Champion: LGB=1.0 XGB=0.0 Pois=0.0
Avg exact score: 29.17%


## 7. Full Summary

In [10]:
print('='*70)
print('FINAL RESULTS SUMMARY')
print('='*70)

print(f'\nLightGBM best params (update lgbm_model.py defaults if improved):')
print(f'  {lgbm_result["best_params"]}')
print(f'  → Avg exact score CV: {lgbm_result["best_avg_acc"]*100:.2f}%')

print(f'\nXGBoost best params (update xgb_model.py defaults if improved):')
print(f'  {xgb_result["best_params"]}')
print(f'  → Avg exact score CV: {xgb_result["best_avg_acc"]*100:.2f}%')

print(f'\nBest ensemble weights:')
print(f'  LGB={best_ensemble["w_lgb"]}, XGB={best_ensemble["w_xgb"]}, Poisson={best_ensemble["w_pois"]}')
print(f'  → Avg exact score CV: {best_ensemble["avg_exact"]*100:.2f}%')

print(f'\nBaseline comparisons (avg across 2014/2018/2022 folds, WITH temporal weights):')
poisson_avg = float(poisson_baseline[poisson_baseline['year']=='avg']['exact'].iloc[0])
lgbm_base_avg = float(lgbm_baseline[lgbm_baseline['year']=='avg']['exact'].iloc[0])
xgb_base_avg = float(xgb_baseline[xgb_baseline['year']=='avg']['exact'].iloc[0])
print(f'  Poisson (default):                 {poisson_avg*100:.2f}%')
print(f'  LightGBM (default):                {lgbm_base_avg*100:.2f}%')
print(f'  XGBoost  (default):                {xgb_base_avg*100:.2f}%')
print(f'  LightGBM (tuned, 300 trials):      {lgbm_result["best_avg_acc"]*100:.2f}%')
print(f'  XGBoost  (tuned, 300 trials):      {xgb_result["best_avg_acc"]*100:.2f}%')
print(f'  Ensemble (best weights):           {best_ensemble["avg_exact"]*100:.2f}%')
print(f'\n  Previous best (nb07, no decay):    30.21%')
delta = best_ensemble['avg_exact']*100 - 30.21
print(f'  Delta vs previous best:            {delta:+.2f}%')

FINAL RESULTS SUMMARY

LightGBM best params (update lgbm_model.py defaults if improved):
  {'n_estimators': 461, 'max_depth': 12, 'learning_rate': 0.025037651451202188, 'num_leaves': 200, 'min_child_samples': 44, 'subsample': 0.8033348792290621, 'colsample_bytree': 0.6203538805551276, 'reg_alpha': 7.910467580693931e-06, 'reg_lambda': 0.03337081102245976}
  → Avg exact score CV: 29.17%

XGBoost best params (update xgb_model.py defaults if improved):
  {'n_estimators': 480, 'max_depth': 12, 'learning_rate': 0.015763210575425127, 'subsample': 0.8995970132570389, 'colsample_bytree': 0.6946673112765889, 'gamma': 4.126453395664713, 'min_child_weight': 19, 'reg_alpha': 0.012107938073992418, 'reg_lambda': 4.164462463380124e-06}
  → Avg exact score CV: 28.12%

Best ensemble weights:
  LGB=1.0, XGB=0.0, Poisson=0.0
  → Avg exact score CV: 29.17%

Baseline comparisons (avg across 2014/2018/2022 folds, WITH temporal weights):
  Poisson (default):                 20.83%
  LightGBM (default):       

## 8. 2022 WC Match-by-Match Backtest

In [11]:
from src.models.score_conversion import top_scores as get_top_scores

X_train_22, y_train_22, X_val_22, y_val_22 = cv_splits[2022]
w_train_22 = weights_per_fold[2022]

wc22_mask = (
    (df['tournament_year'] == 2022) &
    (df['competition'].str.strip().str.lower() == 'world cup')
)
wc22_meta = df[wc22_mask][['date', 'team_A', 'team_B']].reset_index(drop=True)

W_LGB  = best_ensemble['w_lgb']
W_XGB  = best_ensemble['w_xgb']
W_POIS = best_ensemble['w_pois']

print(f'Ensemble weights: LGB={W_LGB}, XGB={W_XGB}, Poisson={W_POIS}')

models_ens, wts_ens = [], []
if W_LGB > 0:
    m = LGBMGoalModel(**lgbm_result['best_params'])
    m.fit(X_train_22, y_train_22, sample_weight=w_train_22)
    models_ens.append(m); wts_ens.append(W_LGB)
if W_XGB > 0:
    m = XGBGoalModel(**xgb_result['best_params'])
    m.fit(X_train_22, y_train_22, sample_weight=w_train_22)
    models_ens.append(m); wts_ens.append(W_XGB)
if W_POIS > 0:
    m = PoissonGoalModel()
    m.fit(X_train_22, y_train_22, sample_weight=w_train_22)
    models_ens.append(m); wts_ens.append(W_POIS)

ensemble_model = EnsembleGoalModel(models_ens, weights=wts_ens)
y_pred_22 = np.clip(ensemble_model.predict(X_val_22), 0, None)
y_pred_rounded = np.round(y_pred_22).astype(int)

rows = []
for i in range(len(y_val_22)):
    actual_a, actual_b = int(y_val_22[i, 0]), int(y_val_22[i, 1])
    pred_a, pred_b     = int(y_pred_rounded[i, 0]), int(y_pred_rounded[i, 1])
    lambda_a, lambda_b = float(y_pred_22[i, 0]), float(y_pred_22[i, 1])
    actual_result = 'W' if actual_a > actual_b else ('D' if actual_a == actual_b else 'L')
    pred_result   = 'W' if pred_a   > pred_b   else ('D' if pred_a   == pred_b   else 'L')
    exact_ok      = (actual_a == pred_a) and (actual_b == pred_b)
    result_ok     = actual_result == pred_result
    best_score    = get_top_scores(lambda_a, lambda_b, n=1)[0]
    rows.append({
        'date':      wc22_meta.iloc[i]['date'],
        'team_A':    wc22_meta.iloc[i]['team_A'],
        'team_B':    wc22_meta.iloc[i]['team_B'],
        'actual':    f'{actual_a}-{actual_b}',
        'predicted': f'{pred_a}-{pred_b}',
        'lambda':    f'{lambda_a:.2f}-{lambda_b:.2f}',
        'top_score': f'{best_score[0]}-{best_score[1]} ({best_score[2]*100:.1f}%)',
        'exact':     exact_ok,
        'result':    result_ok,
    })

results_22 = pd.DataFrame(rows)
n_exact  = results_22['exact'].sum()
n_result = results_22['result'].sum()
n_total  = len(results_22)

print(f'\n2022 WC Ensemble Backtest ({n_total} matches)')
print(f'  Exact score:    {n_exact}/{n_total}  ({n_exact/n_total*100:.1f}%)')
print(f'  Result W/D/L:   {n_result}/{n_total}  ({n_result/n_total*100:.1f}%)')
print(f'  Goal MAE (exp): {np.mean(np.abs(y_val_22 - y_pred_22)):.3f}')

print('\nResult breakdown:')
print(f'  Wrong result:           {(~results_22["result"]).sum()} matches')
print(f'  Right result, wrong score: {(results_22["result"] & ~results_22["exact"]).sum()} matches')
print(f'  Right exact score:      {results_22["exact"].sum()} matches')

Ensemble weights: LGB=1.0, XGB=0.0, Poisson=0.0

2022 WC Ensemble Backtest (64 matches)
  Exact score:    19/64  (29.7%)
  Result W/D/L:   60/64  (93.8%)
  Goal MAE (exp): 0.731

Result breakdown:
  Wrong result:           4 matches
  Right result, wrong score: 41 matches
  Right exact score:      19 matches


In [12]:
# Full match-by-match table
pd.set_option('display.max_rows', 70)
pd.set_option('display.width', 130)
print(results_22.sort_values('date')[['date','team_A','team_B','actual','predicted','lambda','top_score','exact','result']].to_string(index=False))

      date        team_A        team_B actual predicted    lambda   top_score  exact  result
2022-11-20         Qatar       Ecuador    0-2       0-3 0.35-2.74 0-2 (17.1%)  False    True
2022-11-21       England          Iran    6-2       2-0 2.38-0.32 2-0 (19.1%)  False    True
2022-11-21 United States         Wales    1-1       1-1 1.02-0.94 1-0 (14.4%)   True    True
2022-11-21   Netherlands       Senegal    2-0       3-1 2.63-0.51 2-0 (14.9%)  False    True
2022-11-22  Saudi Arabia     Argentina    2-1       1-1 0.62-1.10 0-1 (19.7%)  False   False
2022-11-22        Mexico        Poland    0-0       1-1 0.91-1.17 0-1 (14.7%)  False    True
2022-11-22        France     Australia    4-1       2-1 2.42-0.60 2-0 (14.2%)  False    True
2022-11-22       Denmark       Tunisia    0-0       1-1 0.95-0.99 0-0 (14.4%)  False    True
2022-11-23         Spain    Costa Rica    7-0       3-1 2.87-0.62 2-0 (12.5%)  False    True
2022-11-23         Japan       Germany    2-1       2-1 1.98-0.51 1-0 